In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

BRONZE_TABLE = "bronze_flights"
SILVER_TABLE = "silver_flights"

print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")

Reading from : bronze_flights
Writing to   : silver_flights


In [0]:
bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()
print(f"Bronze rows: {bronze_count:,}")
bronze_df.printSchema()

Bronze rows: 309,108
root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: double (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [0]:
business_cols = [
    "Reporting_Airline", "Origin", "Dest", "Route", "ArrDel15",
    "Day", "Month", "DayOfWeek", "IsWeekend",
    "IsFixedHoliday", "IsHolidayWindow",
    "DepHour", "DepTimeCategory"
]

deduped_df = bronze_df.select(*business_cols).dropDuplicates()

deduped_count = deduped_df.count()
duplicates_removed = bronze_count - deduped_count

print(f"Rows before dedup: {bronze_count:,}")
print(f"Rows after  dedup: {deduped_count:,}")
print(f"Duplicates removed: {duplicates_removed:,}")

Rows before dedup: 309,108
Rows after  dedup: 302,192
Duplicates removed: 6,916


In [0]:
invalid_conditions = (
      ~F.col("ArrDel15").isin([0.0, 1.0])
    | ~F.col("Month").between(1, 12)
    | ~F.col("Day").between(1, 31)
    | ~F.col("DayOfWeek").between(1, 7)
    | ~F.col("DepHour").between(0, 23)
    | ~F.col("IsWeekend").isin([0, 1])
    | ~F.col("IsFixedHoliday").isin([0, 1])
    | ~F.col("IsHolidayWindow").isin([0, 1])
    | F.col("Reporting_Airline").isNull()
    | F.col("Origin").isNull()
    | F.col("Dest").isNull()
)

invalid_count = deduped_df.filter(invalid_conditions).count()
print(f"Invalid rows found: {invalid_count:,}")

validated_df = deduped_df.filter(~invalid_conditions)
print(f"Rows after validation: {validated_df.count():,}")

Invalid rows found: 0
Rows after validation: 302,192


In [0]:
typed_df = validated_df.withColumn("ArrDel15", F.col("ArrDel15").cast(IntegerType()))
typed_df.printSchema()

root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)



In [0]:
dated_df = (
    typed_df
        .withColumn("year", F.when(F.col("Month") == 12, F.lit(2024)).otherwise(F.lit(2025)))
        .withColumn("flight_date", F.make_date(F.col("year"), F.col("Month"), F.col("Day")))
        .withColumn(
            "flight_hour_ts",
            F.make_timestamp(
                F.col("year"), F.col("Month"), F.col("Day"),
                F.col("DepHour"), F.lit(0), F.lit(0.0)
            )
        )
)

dated_df.select("year", "Month", "Day", "DepHour", "flight_date", "flight_hour_ts").show(5, truncate=False)

+----+-----+---+-------+-----------+-------------------+
|year|Month|Day|DepHour|flight_date|flight_hour_ts     |
+----+-----+---+-------+-----------+-------------------+
|2025|1    |1  |6      |2025-01-01 |2025-01-01 06:00:00|
|2025|1    |2  |6      |2025-01-02 |2025-01-02 06:00:00|
|2025|1    |3  |6      |2025-01-03 |2025-01-03 06:00:00|
|2025|1    |4  |7      |2025-01-04 |2025-01-04 07:00:00|
|2025|1    |5  |6      |2025-01-05 |2025-01-05 06:00:00|
+----+-----+---+-------+-----------+-------------------+
only showing top 5 rows


In [0]:
dated_df.groupBy("year", "Month").count().orderBy("year", "Month").show(20)

+----+-----+-----+
|year|Month|count|
+----+-----+-----+
|2024|   12|29067|
|2025|    1|23664|
|2025|    2|22243|
|2025|    3|25180|
|2025|    4|25365|
|2025|    5|25645|
|2025|    6|24692|
|2025|    7|25020|
|2025|    8|25864|
|2025|    9|24712|
|2025|   10|26044|
|2025|   11|24696|
+----+-----+-----+



In [0]:
silver_df = (
    dated_df
        .drop("year")
        .withColumn("_silver_ingested_at", F.current_timestamp())
)

silver_df.printSchema()

root
 |-- Reporting_Airline: string (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Route: string (nullable = true)
 |-- ArrDel15: integer (nullable = true)
 |-- Day: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- IsWeekend: integer (nullable = true)
 |-- IsFixedHoliday: integer (nullable = true)
 |-- IsHolidayWindow: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- DepTimeCategory: string (nullable = true)
 |-- flight_date: date (nullable = true)
 |-- flight_hour_ts: timestamp (nullable = true)
 |-- _silver_ingested_at: timestamp (nullable = false)



In [0]:
(
    silver_df.write
             .format("delta")
             .mode("overwrite")
             .option("overwriteSchema", "true")
             .saveAsTable(SILVER_TABLE)
)

print(f"Wrote Delta table: {SILVER_TABLE}")

Wrote Delta table: silver_flights


In [0]:
silver_count = spark.table(SILVER_TABLE).count()
print(f"Silver table row count: {silver_count:,}")

spark.sql(f"""
    SELECT ArrDel15,
           COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM {SILVER_TABLE}
    GROUP BY ArrDel15
    ORDER BY ArrDel15
""").show()

spark.sql(f"""
    SELECT MIN(flight_date) AS earliest_date,
           MAX(flight_date) AS latest_date,
           COUNT(DISTINCT flight_date) AS distinct_dates
    FROM {SILVER_TABLE}
""").show(truncate=False)

spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").show(5, truncate=False)

Silver table row count: 302,192
+--------+------+-----+
|ArrDel15|     n|  pct|
+--------+------+-----+
|       0|228939|75.76|
|       1| 73253|24.24|
+--------+------+-----+

+-------------+-----------+--------------+
|earliest_date|latest_date|distinct_dates|
+-------------+-----------+--------------+
|2024-12-01   |2025-11-30 |365           |
+-------------+-----------+--------------+

+-------+-------------------+--------------+---------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------